# Fast anomaly detection with few-step flow maps

Normalizing flows are fast to evaluate but slow to train; flow matching is the
reverse. The few-step flow maps in `inspectorch.fast_flows` aim at both: they
train like flow matching and evaluate the density in a handful of steps.

This notebook follows `01_normalizing_flow.ipynb` with **FALCON** (a MeanFlow
map plus a cycle-consistency term that keeps it invertible). Swap the `kind`
for `"meanflow"`, `"f2d2"` or `"regflow"` to try the others. To compare all of
them on the same data and budget, use `tools/bench_fast_likelihood.py`.

The training cell is meant for a GPU: on a laptop CPU, reduce `num_epochs`.

In [ ]:
from inspectorch import GeneralizedPatchedDataset
from inspectorch.fast_flows import FastFlowEstimator
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
import torch.utils.data
from einops import rearrange
from scipy.stats import spearmanr
from inspectorch.plot_params import set_params
from inspectorch import utils

In [ ]:
# Suppress some warnings for cleaner output
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# Set matplotlib plotting parameters for consistent style
set_params()

In [ ]:
# Print device info for debugging and reproducibility
utils.device_info()

## Data

In [ ]:
# --- Data loading and preprocessing ---

# Read the compressed Hinode data from file
with np.load("hinode_data.npz") as npzfile:
    data = npzfile["data"] / 186.0
    wav = npzfile["wav"]
    pixel_size = npzfile["pixel_size"] * 3.0

ny, nwav, nx = data.shape
print(f"Data shape: {data.shape}")

In [ ]:
# --- Dataset preparation ---

# Wrap the data in a patched dataset for spectral analysis
dataset = GeneralizedPatchedDataset(
    data,
    dim_names="y wav x",  # Define the dimensions of the dataset
    feature_dims=["wav"],  # Specify the feature dimensions for spectral analysis
)

print(f"Data (patched) shape: {dataset.shape}")
extent = np.array([0, data.shape[2], 0, data.shape[0]]) * pixel_size

## Data preparation

In [ ]:
# --- Data preparation for training ---

args = utils.dot_dict()
args.batch_size = 4096

# Create a DataLoader for efficient batch training
train_loader = torch.utils.data.DataLoader(
    dataset, batch_size=args.batch_size, shuffle=True
)

## Density estimation

A FALCON model learns the average velocity $u(x, a, b)$ of the flow between
times $a$ and $b$ (noise at $t=0$, data at $t=1$), so one network call moves a
sample from $a$ to $b$: $X(x, a, b) = x + (b - a)\,u(x, a, b)$. The density of
a spectrum follows from mapping it to noise in a few such steps and adding the
log-determinant of each step's Jacobian.

In [ ]:
# Build the few-step model
model = FastFlowEstimator(kind="falcon")
model.create_flow(input_size=dataset.flow_dim, hidden_features=256, num_layers=4)

n_params = sum(p.numel() for p in model.model.parameters())
print(f"Total params to optimize: {n_params:,}")

## Training the density estimator

In [ ]:
# Set training hyperparameters
args.learning_rate = 1e-3
args.num_epochs = 15
args.device = "auto"  # CUDA > MPS > CPU

model.train_flow(
    train_loader,
    learning_rate=args.learning_rate,
    num_epochs=args.num_epochs,
    device=args.device,
)
print(f"Trained in {model.train_seconds / 60:.2f} minutes")

In [ ]:
# Plot training loss to monitor convergence
plt.figure()
plt.plot(model.training_loss, ".-")
plt.minorticks_on()
plt.xlabel("Epoch")
plt.ylabel("Loss")

## Evaluating the model

In [ ]:
# Log-probability of every spectrum with 4 map steps
args.steps = 4
t0 = time.perf_counter()
log_prob = model.log_prob(dataset, steps=args.steps, device=args.device)
print(f"{len(log_prob)} spectra in {time.perf_counter() - t0:.2f} s with {args.steps} steps")

# Reshape log_prob to match original spatial dimensions
log_prob_reshaped = rearrange(log_prob, "(y x) -> y x", y=ny, x=nx)

In [ ]:
# Plot spatial distribution of log-probabilities
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
spectral_to_white = utils.create_spectral_to_white_cmap()

im = ax.imshow(
    log_prob_reshaped,
    cmap=spectral_to_white,
    vmin=np.percentile(log_prob_reshaped, 0.1),
    vmax=np.percentile(log_prob_reshaped, 99.9),
    origin="lower",
    interpolation="nearest",
    extent=extent,
)
cb = fig.colorbar(im, ax=ax, pad=0.02, shrink=1.0, aspect=40)
cb.set_label(r"$\log\,p_\phi(\mathbf{x})$")
plt.minorticks_on()
plt.xlabel("X [arcsec]")
plt.ylabel("Y [arcsec]")

In [ ]:
# --- Visualization of least common spectra ---

nplots = 5

# Find indices of least probable spectra
idxs = np.argsort(log_prob)[:nplots]
print(f"Least common spectra indices: {idxs}")

average = torch.mean(dataset.patches, axis=0)
fig, ax = plt.subplots(1, 5, figsize=(20, 4), sharex=True, sharey=True)
for i in range(nplots):
    spectrum = dataset[idxs[i]]
    ax[i].plot(wav / 10, spectrum / torch.max(spectrum), color="C3")
    ax[i].plot(wav / 10, average / torch.max(average), color="black", alpha=0.2)
    ax[i].set_title(r"$\log\,p_\phi(\mathbf{{x}})$: ${0:.1f}$".format(log_prob[idxs[i]]))
    ax[i].text(
        0.5,
        0.1,
        r"$\mathbf{{[x,y]}}$ = ({0}, {1})".format(idxs[i] % nx, idxs[i] // nx),
        ha="center",
        transform=ax[i].transAxes,
    )
    ax[i].set_xlabel("Wavelength [nm]")
ax[0].set_ylabel("Intensity [a.u.]")
for axes in ax:
    axes.ticklabel_format(style="plain", axis="x", useOffset=False)

## Speed against number of steps

More steps cost more but follow the flow more closely. The rarest spectra are
what matters here, so each setting is compared with the 8-step ranking: the
share of its 100 rarest spectra that are also among the 8-step 100 rarest.

In [ ]:
results = {}
for steps in (1, 2, 4, 8):
    t0 = time.perf_counter()
    lp = model.log_prob(dataset, steps=steps, device=args.device)
    results[steps] = (lp, time.perf_counter() - t0)

reference = results[8][0]
top_ref = set(np.argsort(reference)[:100])
for steps, (lp, seconds) in results.items():
    overlap = len(set(np.argsort(lp)[:100]) & top_ref) / 100
    rho = spearmanr(lp, reference).statistic
    print(f"{steps} steps: {len(lp) / seconds:9.0f} spectra/s, "
          f"top-100 overlap with 8 steps {overlap:.2f}, Spearman {rho:.3f}")